In [ ]:
import sqlite3
import pandas as pd
import math
import numpy as np
import tensorflow as T
import rdkit
from rdkit.Chem import AllChem
from rdkit import Chem, DataStructs
from rdkit.Chem import rdmolops
from rdkit.Avalon import pyAvalonTools
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem import Descriptors
import numpy
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.metrics import confusion_matrix, classification_report
from sklearn import metrics
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import random
import torch
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint

from chemprop import data, featurizers, models, nn

import os
# Suppress TensorFlow info and warning messages (0 = all logs, 1 = no INFO, 2 = no WARNING, 3 = no ERROR)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'

# Suppress underlying Abseil/gRPC logging library warnings
os.environ["GRPC_VERBOSITY"] = "ERROR"
os.environ["GLOG_minloglevel"] = "2"

import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print("Num GPUs Available: ", len(gpus))
print("GPU Details: ", gpus)

In [ ]:
connection = sqlite3.connect('/Work/chembl_35_sqlite/chembl_35.db')

In [ ]:
crsr = connection.cursor()

In [ ]:
sql_chembl_name = np.asarray(pd.read_csv('/share/My Documents/Extrapolation/chembl_data1.csv'))

In [ ]:
print(sql_chembl_name[2,:])

In [ ]:
print(sql_chembl_name.shape[0])

In [ ]:
def desriptors_calc(data_set):
    train_rdkit = []

    for x in range(data_set.shape[0]):
        counter = 0
        mol = Chem.MolFromSmiles(data_set[x])
        if mol is None: continue
        des = Chem.Descriptors.CalcMolDescriptors(mol)
        cleaned_values = []
        for val in des.values():

            if isinstance(val, (int, float)) and abs(val) > 1e6:
                cleaned_values.append(0.0)
            else:
                cleaned_values.append(val)
                
        train_rdkit.append(cleaned_values)        
    des_res =numpy.asarray(train_rdkit)
    des_res[numpy.isnan(des_res)] = 0
    des_res[numpy.isinf(des_res)] = 0
    return des_res

In [ ]:
def modeling_set_chemprop(x_train, y_train, x_test, y_test, x_val, y_val):
    y_train = y_train.reshape(-1, 1) 
    y_test = y_test.reshape(-1, 1) 
    y_val = y_val.reshape(-1, 1) 
    all_data = [data.MoleculeDatapoint.from_smi(smi, y) for smi, y in zip(x_train, y_train)]
    mols = [d.mol for d in all_data]  # RDkit Mol objects are use for structure based splits
    train_indices, val_indices, test_indices = data.make_split_indices(mols, "random", (0.8, 0.1, 0.1))  # unpack the tuple into three separate lists
    train_data, val_data, test_data = data.split_data_by_indices(all_data, train_indices, val_indices, test_indices)
    all_data_test = [data.MoleculeDatapoint.from_smi(smi, y) for smi, y in zip(x_test, y_test)]
    all_data_val = [data.MoleculeDatapoint.from_smi(smi, y) for smi, y in zip(x_val, y_val)]
    featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
    train_dset = data.MoleculeDataset(train_data[0], featurizer)
    #scaler = train_dset.normalize_targets()
    val_dset = data.MoleculeDataset(val_data[0], featurizer)
    #val_dset.normalize_targets(scaler)
    test_dset = data.MoleculeDataset(test_data[0], featurizer)
    train_loader = data.build_dataloader(train_dset, num_workers=5)
    val_loader = data.build_dataloader(val_dset, num_workers=5, shuffle=False)
    test_loader = data.build_dataloader(test_dset, num_workers=5, shuffle=False)
    
    test_test_dset = data.MoleculeDataset(all_data_test, featurizer=featurizer)
    test_val_dset = data.MoleculeDataset(all_data_val, featurizer=featurizer)
    test_test_loader = data.build_dataloader(test_test_dset, shuffle=False)
    test_val_loader = data.build_dataloader(test_val_dset, shuffle=False)
    
    return train_loader, val_loader, test_loader, test_test_loader, test_val_loader, scaler



In [ ]:
def desriptors_extrapolation(y_train, des_res):
    y_train_X1 = numpy.zeros((y_train.shape[0],y_train.shape[0]))
    for i in range (y_train.shape[0]):
        for k in range (y_train.shape[0]):
                y_train_X1[i,k] = y_train[i] - y_train[k]
    X_train_X1 = numpy.zeros((y_train.shape[0],y_train.shape[0],des_res.shape[1]))
    for i in range (y_train.shape[0]):
        for k in range (y_train.shape[0]):
                X_train_X1[i,k,:] = des_res[i,:] - des_res[k,:]
    return y_train_X1, X_train_X1

In [ ]:
def desriptors_extrapolation_test(x_test, y_train, des_test, des_res):    
    X_train_T1 = numpy.zeros((x_test.shape[0],y_train.shape[0],des_res.shape[1]))
    for i in range (x_test.shape[0]):
        for k in range (y_train.shape[0]):
                X_train_T1[i,k,:] = des_test[i,:] - des_res[k,:]
    return X_train_T1

In [ ]:
def prediction_extrapolation(y_test, y_train, pred):    
    y_train_T1 = numpy.zeros((y_test.shape[0],y_train.shape[0]))
    for i in range (y_test.shape[0]):
        for k in range (y_train.shape[0]):
                y_train_T1[i,k] = pred[i,k] + y_train[k]
    pred_extra = numpy.average(y_train_T1, axis=1)            
    return pred_extra

In [ ]:
def prediction_extrapolation_q(y_test, y_train, pred):    
    y_train_T1 = numpy.zeros((y_test.shape[0],y_train.shape[0]))
    for i in range (y_test.shape[0]):
        for k in range (y_train.shape[0]):
                y_train_T1[i,k] = pred[i,k] + y_train[k]
    pred_extra = numpy.quantile(y_train_T1, 0.1, axis=1)            
    return pred_extra

In [ ]:
def euclidean_distance(a, b):
    return numpy.sqrt(numpy.sum((a - b)**2))

In [ ]:
def prediction_extrapolation_kNN(y_test, y_train, des_test, des_res, pred):    
    y_train_T1 = numpy.zeros((y_test.shape[0]))
    n=3
    for i in range (y_test.shape[0]):
        y = 0
        ex = 0
        sim=numpy.zeros((y_train.shape[0]))
        for k in range (y_train.shape[0]):
                distance = euclidean_distance(des_test[i,:],des_res[k,:])
                sim[k]=distance

        n_indices = numpy.argsort(sim)[:n]
        res=0
        for b in range (n_indices.shape[0]):
            res = res + pred[i,n_indices[b]] + y_train[n_indices[b]]
        y_train_T1[i]=res/n    
    return y_train_T1

In [ ]:
def model_build(X_train_X1, y_train_X1):
    T.keras.backend.clear_session()
    desc1 = T.keras.Input(shape=[y_train.shape[0],des_res.shape[1]],name='desc1')
    x = T.keras.layers.Dense(500, activation='relu')(desc1)
    x = T.keras.layers.Dense(50, activation='relu')(x)
    out1 = T.keras.layers.Dense(1, activation='linear', name='out1')(x)
    model = T.keras.Model(inputs=[desc1], outputs=[out1])
    model.compile(loss=T.keras.losses.MeanSquaredError(), optimizer=T.keras.optimizers.Adam(0.0001))
    model.fit([X_train_X1], [y_train_X1], verbose=0, epochs=100, batch_size=20)
    return model

In [ ]:
def model_build_regular(des_res, y_train):    
    T.keras.backend.clear_session()
    desc1 = T.keras.Input(shape=[des_res.shape[1]],name='desc1')
    x = T.keras.layers.Dense(500, activation='relu')(desc1)
    x = T.keras.layers.Dense(50, activation='relu')(x)
    out1 = T.keras.layers.Dense(1, activation='linear', name='out1')(x)
    model2 = T.keras.Model(inputs=[desc1], outputs=[out1])
    model2.compile(loss=T.keras.losses.MeanSquaredError(), optimizer=T.keras.optimizers.Adam(0.0001))
    model2.fit([des_res], [y_train], verbose=0, epochs=100, batch_size=20)
    return model2

In [ ]:
def model_build_Chemprop(scaler):
    mp = nn.BondMessagePassing()
    agg = nn.MeanAggregation()
    #output_transform = nn.UnscaleTransform.from_standard_scaler(scaler)
    ffn = nn.RegressionFFN()
    batch_norm = False
    metric_list = [nn.metrics.RMSE(), nn.metrics.MAE()]
    mpnn = models.MPNN(mp, agg, ffn, batch_norm, metric_list)
    checkpointing = ModelCheckpoint(
        "/share/My Documents/Extrapolation/checkpoints",  # Directory where model checkpoints will be saved
        "best-{epoch}-{val_loss:.2f}",  # Filename format for checkpoints, including epoch and validation loss
        "val_loss",  # Metric used to select the best checkpoint (based on validation loss)
        mode="min",  # Save the checkpoint with the lowest validation loss (minimization objective)
        save_last=True,  # Always save the most recent checkpoint, even if it's not the best
        )
    trainer = pl.Trainer(
        logger=False,
        enable_checkpointing=True, # Use `True` if you want to save model checkpoints. The checkpoints will be saved in the `checkpoints` folder.
        enable_progress_bar=False,
        accelerator="gpu",
        devices=1,
        max_epochs=20, # number of epochs to train for
        callbacks=[checkpointing], # Use the configured checkpoint callback
    )    
    
    
    return trainer, mpnn


In [ ]:
def save_results(name_id, tips):
    save_path = f'/share/My Documents/Extrapolation/{name_id}.txt'
    df = pd.DataFrame(tips)
    df.to_csv(save_path, sep='\t', index=False, header=False)

In [ ]:
sql_command = """ SELECT m.chembl_id AS compound_chembl_id,   
s.canonical_smiles,   
r.compound_key,   
a.description AS assay_description,
act.standard_type,
a.confidence_score,
a.assay_id,   
act.standard_relation,   
act.standard_value,   
act.standard_units,   
act.activity_comment, 
act.activity_id
FROM compound_structures s
 JOIN molecule_dictionary m on s.molregno = m.molregno 
 JOIN compound_records r on m.molregno = r.molregno  
 JOIN docs d on r.doc_id = d.doc_id 
 JOIN activities act on r.record_id = act.record_id
 JOIN assays a on act.assay_id = a.assay_id 
 JOIN target_dictionary t on a.tid = t.tid 
AND t.chembl_id = '{x}'
AND a.chembl_id = '{y}'
AND act.standard_type = 'IC50'
AND a.confidence_score >= 7
AND act.standard_relation = '='
AND act.standard_units = 'nM';"""

In [ ]:
db_path = '/Work/chembl_35_sqlite/chembl_35.db'
connection = sqlite3.connect(db_path)
crsr = connection.cursor()
sql=sql_command.format(x=sql_chembl_name[0,1], y=sql_chembl_name[0,0])
crsr.execute(sql)
ans = crsr.fetchall()
print(ans)
dataset = pd.DataFrame(ans)
print(dataset[8])
#dataset = dataset[dataset[8] > 0]


In [ ]:
for i in range(sql_chembl_name.shape[0]):
#for i in range(3):
    sql=sql_command.format(x=sql_chembl_name[i,1], y=sql_chembl_name[i,0])
    #print(sql)
    crsr.execute(sql)
    ans = crsr.fetchall()
    dataset = pd.DataFrame(ans)
    print(dataset)
    dataset = dataset[dataset[8] > 0]
    dataset["pIC50"] = np.log10(dataset[8] * 1e-9)
    dataset["SD"] = dataset.groupby(0)["pIC50"].transform(np.std)
    df_filtered = dataset[(dataset["SD"] <= 0.5) | (dataset["SD"].isna())]
    df_final = (
        df_filtered.groupby(0)
        .agg(
            {
                1: "first",  # Keeps canonical_smiles string from column index 1
                "pIC50": "mean",  # Averages the calculated pIC50 log values
            }
        )
        .reset_index()
    )
    dataset = df_final.sort_values(by="pIC50")
    num_rows = int(len(dataset)*0.1)
    valid = dataset.iloc[:num_rows]
    db = dataset.iloc[num_rows:]
    test_mask = (np.arange(len(db)) % 10) < 3
    test_df = db[test_mask]
    train_df = db[~test_mask]
    train_df = train_df.sample(frac=1, random_state=1)
    test_df = test_df.sample(frac=1, random_state=1)    

    #train_df = db.sample(frac=0.7, random_state = 1)
    #test_df = db.drop(train_df.index)
    print(len(valid))
    print(len(db))
    print(len(dataset))
    train_df["Set"] = 'train'
    test_df["Set"] = 'test'
    valid["Set"] = 'external_validation'
    train_df["Target_ID"] = sql_chembl_name[i,1]
    test_df["Target_ID"] = sql_chembl_name[i,1]
    valid["Target_ID"] = sql_chembl_name[i,1]
    combined_df = pd.concat([train_df, test_df, valid], ignore_index=True)
    name_id_raw_data = '/share/My Documents/Extrapolation/Raw_data/'+str(sql_chembl_name[i,1]+'_raw_data.csv')
    combined_df.to_csv(name_id_raw_data, index=False)
    y_train = train_df.iloc[:,2].values
    x_train = train_df.iloc[:,1].values
    y_test = test_df.iloc[:,2].values
    x_test = test_df.iloc[:,1].values
    y_val = valid.iloc[:,2].values
    x_val = valid.iloc[:,1].values
    des_res =desriptors_calc(x_train)
    #scaler = MinMaxScaler()
    scaler = StandardScaler()
    des_res = scaler.fit_transform(des_res)
    y_train_X1, X_train_X1 = desriptors_extrapolation(y_train, des_res)
    
    model = model_build(X_train_X1, y_train_X1)
    des_test =desriptors_calc(x_test)
    des_test = scaler.transform(des_test)
    outlier_mask = np.abs(des_test) > 6.0
    num_outliers = np.sum(outlier_mask)
    des_test[outlier_mask] = 0.0
    X_train_T1 = desriptors_extrapolation_test(x_test, y_train, des_test, des_res)
    #Test    
    predictions = model.predict([X_train_T1], verbose=0)
    pred = predictions[:,:,0]
    pred_extra = prediction_extrapolation(y_test, y_train, pred)
    pred_extra_q = prediction_extrapolation_q(y_test, y_train, pred)
    pred_extra_kNN = prediction_extrapolation_kNN(y_test, y_train, des_test, des_res, pred)
    #Average
    all_data = numpy.stack((y_test, pred_extra), axis=-1)
    name_id = str(sql_chembl_name[i,1]+"_test")
    save_results(name_id,all_data)
    #Quantile
    all_data_q = numpy.stack((y_test, pred_extra_q), axis=-1)
    name_id = str(sql_chembl_name[i,1]+"_q_test")
    save_results(name_id,all_data_q)
    #kNN
    all_data_kNN = numpy.stack((y_test, pred_extra_kNN), axis=-1)
    name_id = str(sql_chembl_name[i,1]+"_test_kNN")
    save_results(name_id,all_data_kNN)
    #Validation    
    des_val =desriptors_calc(x_val)
    des_val = scaler.transform(des_val)
    outlier_mask = np.abs(des_val) > 6.0
    num_outliers = np.sum(outlier_mask)
    des_val[outlier_mask] = 0.0

    X_train_Val = desriptors_extrapolation_test(x_val, y_train, des_val, des_res)
    predictions_val = model.predict([X_train_Val], verbose=0)
    pred_val = predictions_val[:,:,0]
    pred_extra_val = prediction_extrapolation(y_val, y_train, pred_val)
    pred_extra_val_q = prediction_extrapolation_q(y_val, y_train, pred_val)
    pred_extra_val_kNN = prediction_extrapolation_kNN(y_val, y_train, des_test, des_res, pred_val)
    #Average
    all_data_val = numpy.stack((y_val, pred_extra_val), axis=-1)
    name_id_val = str(sql_chembl_name[i,1]+"_val")
    save_results(name_id_val,all_data_val)
    #Quantile
    all_data_val_q = numpy.stack((y_val, pred_extra_val_q), axis=-1)
    name_id_val = str(sql_chembl_name[i,1]+"_q_val")
    save_results(name_id_val,all_data_val_q)
    #kNN
    all_data_val_kNN = numpy.stack((y_val, pred_extra_val_kNN), axis=-1)
    name_id_val = str(sql_chembl_name[i,1]+"_val_kNN")
    save_results(name_id_val,all_data_val_kNN)
    #del model
    #Regular
    model_regular = model_build_regular(des_res, y_train)
    predictions_test_regular = model_regular.predict([des_test], verbose=0)
    predictions_val_regular = model_regular.predict([des_val], verbose=0)
    pred_test_regular = predictions_test_regular[:,0]
    pred_val_regular = predictions_val_regular[:,0]
    all_data_test_regular = numpy.stack((y_test, pred_test_regular), axis=-1)
    all_data_val_regular = numpy.stack((y_val, pred_val_regular), axis=-1)
    name_id_test_regular = str(sql_chembl_name[i,1]+"_test_regular")
    save_results(name_id_test_regular,all_data_test_regular)
    name_id_val_regular = str(sql_chembl_name[i,1]+"_val_regular")
    save_results(name_id_val_regular,all_data_val_regular)
    #del model_regular   

    #Chemprop
    train_loader, val_loader, test_loader, test_test_loader, test_val_loader, scaler = modeling_set_chemprop(x_train, y_train, x_test, y_test, x_val, y_val)
    
    model_chemprop, mpnn = model_build_Chemprop(scaler)
    model_chemprop.fit(mpnn, train_loader, val_loader)
    with torch.inference_mode():
        trainer = pl.Trainer(
            logger=None,
            enable_progress_bar=False,
            accelerator="gpu",
            devices=1
        )
        predictions_test_chemprop = trainer.predict(mpnn, test_test_loader)   
        predictions_val_chemprop = trainer.predict(mpnn, test_val_loader)   
    pred_test_chemprop = numpy.asarray(np.concatenate(predictions_test_chemprop, axis=0))
    pred_val_chemprop = numpy.asarray(np.concatenate(predictions_val_chemprop, axis=0))

    all_data_test_chemprop = numpy.stack((y_test, pred_test_chemprop[:,0]), axis=-1)
    all_data_val_chemprop = numpy.stack((y_val, pred_val_chemprop[:,0]), axis=-1)
    name_id_test_chemprop = str(sql_chembl_name[i,1]+"_test_chemprop")
    save_results(name_id_test_chemprop,all_data_test_chemprop)
    name_id_val_chemprop = str(sql_chembl_name[i,1]+"_val_chemprop")
    save_results(name_id_val_chemprop,all_data_val_chemprop)
    del model, model_regular, model_chemprop, mpnn
    print("curent_iteration is # "+str(i))
    